In [ ]:
import joblib
import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
data = np.load('modelling_data.npz')
Xraw = data['Xraw']
yraw = data['yraw']
#yraw_safe = np.clip(data['yraw'], a_min=1e-30, a_max=None)
#yraw = np.log10(yraw_safe)
Xfit = data['Xfit']
yfit = data['yfit']
#yfit_safe = np.clip(data['yfit'], a_min=1e-30, a_max=None)
#yfit = np.log10(yfit_safe)
print(Xraw.shape, yraw.shape, Xfit.shape, yfit.shape)

In [ ]:
v1 = 10
v2 = 0

selectedindex = np.where((Xraw[:, 0] == v1) & (Xraw[:, 1] == v2))
Xraw_selected = Xraw[selectedindex]
yraw_selected = yraw[selectedindex]
print(Xraw_selected.shape, yraw_selected.shape)
j1s_raw = Xraw_selected[:, 2]
j2s_raw = Xraw_selected[:, 3]
es_raw = Xraw_selected[:, 4]
selectedindex = np.where((Xfit[:, 0] == v1) & (Xfit[:, 1] == v2))
Xfit_selected = Xfit[selectedindex]
yfit_selected = yfit[selectedindex]
print(Xfit_selected.shape, yfit_selected.shape)     
j1s_fit = Xfit_selected[:, 2]
j2s_fit = Xfit_selected[:, 3]
es_fit = Xfit_selected[:, 4]
# remove the first two soloumn which are v1 and v2
Xraw_selected = Xraw_selected[:, 2:]
Xfit_selected = Xfit_selected[:, 2:]
print(Xraw_selected.shape, Xfit_selected.shape)

In [ ]:
# testplot 
j1 = j1s_raw[0]
j2 = j2s_raw[0]

selectedindex = np.where((Xraw_selected[:, 0] == j1) & (Xraw_selected[:, 1] == j2))
xraw = Xraw_selected[selectedindex]
yraw = yraw_selected[selectedindex]
print(xraw.shape, yraw.shape)
es = xraw[:, 2]

selectedindex = np.where((Xfit_selected[:, 0] == j1) & (Xfit_selected[:, 1] == j2))
xfit = Xfit_selected[selectedindex]
yfit = yfit_selected[selectedindex]
print(xfit.shape, yfit.shape)
esfit = xfit[:, 2]

plt.scatter(es, yraw, marker='o', color='blue', label='raw')  
# plot as a line plot
plt.plot(esfit, yfit, marker='.', color='red', label='fit')
plt.xlabel('es')
plt.ylabel('log10(y)')
plt.title(f'j1={j1}, j2={j2}')
plt.legend()
plt.show()


In [ ]:
# Split fitted data for Pre-Training
Xfit_selected_train, Xfit_selected_test, yfit_selected_train, yfit_selected_test = train_test_split(
        Xfit_selected, yfit_selected, test_size=0.2, random_state=42
    )

# Split raw data for Fine-Tuning
Xraw_selected_train, Xraw_selected_test, yraw_selected_train, yraw_selected_test = train_test_split(
        Xraw_selected, yraw_selected, test_size=0.2, random_state=42
    )
print("Training shapes:")
print(Xfit_selected_train.shape, yfit_selected_train.shape)
print(Xraw_selected_train.shape, yraw_selected_train.shape)
print("Test shapes:")
print(Xfit_selected_test.shape, yfit_selected_test.shape)
print(Xraw_selected_test.shape, yraw_selected_test.shape)

In [ ]:
scalerXraw = StandardScaler()
Xraw_selected_train_scaled = scalerXraw.fit_transform(Xraw_selected_train)
Xraw_selected_test_scaled = scalerXraw.transform(Xraw_selected_test)
scaleryraw = StandardScaler()
yraw_selected_train_scaled = scaleryraw.fit_transform(yraw_selected_train.reshape(-1, 1)).flatten()
yraw_selected_test_scaled = scaleryraw.transform(yraw_selected_test.reshape(-1, 1)).flatten()
scalerXfit = StandardScaler()
Xfit_selected_train_scaled = scalerXfit.fit_transform(Xfit_selected_train)
Xfit_selected_test_scaled = scalerXfit.transform(Xfit_selected_test)
scaleryfit = StandardScaler()
yfit_selected_train_scaled = scaleryfit.fit_transform(yfit_selected_train.reshape(-1, 1)).flatten()
yfit_selected_test_scaled = scaleryfit.transform(yfit_selected_test.reshape(-1, 1)).flatten()       


In [ ]:
def build_model(input_dim, shapes=[512, 
                                   'BN', 
                                   256, 
                                   'BN', 
                                   256, 
                                   128, 
                                   64]):
    model = models.Sequential()
    model.add(layers.InputLayer(shape=(input_dim,)))
    for shape in shapes:
        if shape == 'BN':
            model.add(layers.BatchNormalization())
        else:
            model.add(layers.Dense(shape, activation='relu'))
    model.add(layers.Dense(1))  # Single linear output for regression (cross-section)
    return model

In [ ]:

shapes=[128, 'BN', 34, 'BN', 32, 12]
model = build_model(Xfit_selected_train_scaled.shape[1])

print("\n--- PHASE 1: Pre-training on Fitted Data ---")
# Standard learning rate for initial training
model.compile(optimizer=optimizers.Adam(learning_rate=0.001),
               loss='mse')
    
early_stop = EarlyStopping(
        monitor='val_loss',         # The metric to watch (Validation MSE)
        patience=5,                 # How many epochs to wait before stopping if no improvement
        min_delta=1e-6,             # Minimum change required to count as an "improvement"
        restore_best_weights=True,  # CRITICAL: Reverts the model to its best state!
        verbose=1                   # Prints a message when early stopping is triggered
    )

history_fit = model.fit(
        Xfit_selected_train_scaled, yfit_selected_train_scaled,
        validation_data=(Xfit_selected_test_scaled, yfit_selected_test_scaled),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )

In [ ]:
# plot hisytory
plt.figure(figsize=(10, 6))
plt.plot(history_fit.history['loss'], label='Training Loss (Fitted)', color='blue')
plt.plot(history_fit.history['val_loss'], label='Validation Loss (Fitted)', color='orange')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Pre-Training on Fitted Data')
plt.legend()
plt.show()

In [ ]:
print("\n--- PHASE 2: Fine-tuning on Raw Data ---")
# Lower the learning rate drastically so we don't destroy the physics we just learned
model.compile(optimizer=optimizers.Adam(learning_rate=0.0001), loss='mse')

history_raw = model.fit(
        Xraw_selected_train_scaled, yraw_selected_train,
        validation_data=(Xraw_selected_test_scaled, yraw_selected_test),
        epochs=200,
        batch_size=64, # Smaller batch size for smaller dataset
        callbacks=[early_stop],
        verbose=1
    )


In [ ]:
# plot history
plt.figure(figsize=(10, 6))
plt.plot(history_raw.history['loss'], label='Training Loss (Raw)', color='green')
plt.plot(history_raw.history['val_loss'], label='Validation Loss (Raw)', color='red')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Fine-Tuning on Raw Data')
plt.legend()
plt.show()

In [ ]:
# predict for training and test and plot
yfit_train_pred_scaled = model.predict(Xfit_selected_train_scaled).flatten()
yfit_test_pred_scaled = model.predict(Xfit_selected_test_scaled).flatten()
yfit_train_pred = scaleryfit.inverse_transform(yfit_train_pred_scaled.reshape(-1, 1)).flatten()
yfit_test_pred = scaleryfit.inverse_transform(yfit_test_pred_scaled.reshape(-1, 1)).flatten()
rmse_fit_train = np.sqrt(np.mean((yfit_train_pred - yfit_selected_train)**2))
mape_fit_train = np.mean(np.abs((yfit_selected_train - yfit_train_pred) / yfit_selected_train)) * 100
rmse_fit_test = np.sqrt(np.mean((yfit_test_pred - yfit_selected_test)**2))
mape_fit_test = np.mean(np.abs((yfit_selected_test - yfit_test_pred) / yfit_selected_test)) * 100
print(f"Fitted Data - Train RMSE: {rmse_fit_train:.4f}, Test RMSE: {rmse_fit_test:.4f}")
print(f"Fitted Data - Train MAPE: {mape_fit_train:.2f}%, Test MAPE: {mape_fit_test:.2f}%")
yraw_train_pred_scaled = model.predict(Xraw_selected_train_scaled).flatten()
yraw_test_pred_scaled = model.predict(Xraw_selected_test_scaled).flatten()
yraw_train_pred = scaleryraw.inverse_transform(yraw_train_pred_scaled.reshape(-1, 1)).flatten()
yraw_test_pred = scaleryraw.inverse_transform(yraw_test_pred_scaled.reshape(-1, 1)).flatten()   
rmse_raw_train = np.sqrt(np.mean((yraw_train_pred - yraw_selected_train)**2))
mape_raw_train = np.mean(np.abs((yraw_selected_train - yraw_train_pred) / yraw_selected_train)) * 100
rmse_raw_test = np.sqrt(np.mean((yraw_test_pred - yraw_selected_test)**2))
mape_raw_test = np.mean(np.abs((yraw_selected_test - yraw_test_pred) / yraw_selected_test)) * 100
print(f"Raw Data - Train RMSE: {rmse_raw_train:.4f}, Test RMSE: {rmse_raw_test:.4f}")
print(f"Raw Data - Train MAPE: {mape_raw_train:.2f}%, Test MAPE: {mape_raw_test:.2f}%")


In [ ]:
# scatter plot for fitted data
plt.figure(figsize=(10, 6))
plt.scatter(yfit_selected_train, yfit_train_pred, label='Train', color='blue', alpha=0.5)
plt.scatter(yfit_selected_test, yfit_test_pred, label='Test', color='orange', alpha=0.5)
plt.xlabel('True log10(Cross-Section)')
plt.ylabel('Predicted log10(Cross-Section)')
plt.title('Predicted vs True for Fitted Data')
plt.legend()
plt.show()